# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [1]:
import langchain_community
import langchain
import openai
import langchain_openai
print(f"LangChain version: {langchain.__version__}")
print(f"LangChain Community version: {langchain_community.__version__}")
print(f"OpenAI version: {openai.__version__}")

LangChain version: 1.2.7
LangChain Community version: 0.4.1
OpenAI version: 2.15.0


In [17]:
from langchain_classic.chains import RetrievalQA
from langchain_classic.chat_models import ChatOpenAI
from langchain_classic.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import Tool
from langchain_classic import hub
from openai import AzureOpenAI
from IPython.display import Markdown, display

import os
import tiktoken
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

In [3]:
load_dotenv()

True

In [4]:
# Load GPT-4 model
llm = ChatOpenAI(model="gpt-4", temperature=0)

# Tokenizer for GPT-4
encoding = tiktoken.encoding_for_model("gpt-4")

# Tavily instance
tavily_search = TavilySearchResults()

# Define safe search function (trim to 3500 tokens)
def safe_search(query: str) -> str:
    """
    Performs a Tavily web search and ensures the result is short enough
    to fit within the GPT-4 token limit.
    """
    result = tavily_search.invoke({"query": query})  # Use invoke instead of run
    
    # Handle result format
    if isinstance(result, list):
        result_text = " ".join([str(item) for item in result])
    elif isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)
    
    # Count and trim tokens
    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  
    return trimmed

# Define the tool
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Web search tool that trims output to avoid token overflow with GPT-4"
    )
]

# Get the React prompt from hub
prompt = hub.pull("hwchase17/react")

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# Run the agent
response = agent_executor.invoke({"input": "What is the latest AI research on retrieval-augmented generation"})
print(response["output"])



> Entering new AgentExecutor chain...
I need to find the most recent research on retrieval-augmented generation in the field of AI. I can use the TavilySafeSearch tool to find this information.
Action: TavilySafeSearch
Action Input: Latest AI research on retrieval-augmented generation{'title': 'What Is Retrieval-Augmented Generation aka RAG - NVIDIA Blog', 'url': 'https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation/', 'content': 'The paper, with coauthors from the former Facebook AI Research (now Meta AI), University College London and New York University, called RAG “a general-purpose fine-tuning recipe” because it can be used by nearly any LLM to connect with practically any external resource.\n\n## Building User Trust\n\nRetrieval-augmented generation gives models sources they can cite, like footnotes in a research paper, so users can check any claims. That builds trust.\n\nWhat’s more, the technique can help models clear up ambiguity in a user query. It also redu

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [13]:
# Your prompt here
prompt_1 = """
Research and summarize the latest advancements in generative AI in healthcare for 2025.
"""


# Run it
response_1 = agent_executor.invoke({"input": prompt_1})




> Entering new AgentExecutor chain...
I need to find the latest advancements in generative AI in healthcare for 2025. Since it's a future date, I should look for predictions, ongoing research, and expected advancements in this field.
Action: TavilySafeSearch
Action Input: Latest advancements in generative AI in healthcare for 2025{'title': 'Generative AI in clinical (2020–2025): a mini-review of applications ...', 'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12620437/', 'content': "Generative artificial intelligence (G-AI) has moved from proof-of-concept demonstrations to practical tools that augment radiology, dermatology, genetics, drug discovery, and electronic-health-record analysis. This mini-review synthesizes fifteen studies published between 2020 and 2025 that collectively illustrate three dominant trends: data augmentation for imbalanced or privacy-restricted datasets, automation of expert-intensive tasks such as radiology reporting, and generation of new biomedical know

In [19]:
display(Markdown(response_1["output"]))


The latest advancements in generative AI in healthcare for 2025 include its use in radiology, dermatology, genetics, drug discovery, and electronic health record analysis. It is being used for data augmentation, automation of expert-intensive tasks, and generation of new biomedical knowledge. It is also being used to handle tasks like medical transcription, scheduling, and data entry, and to streamline administrative workloads and support clinical decision-making. In drug discovery, it is being used to speed up the analysis of candidates, simulate their interactions with the human body, and deliver new, more affordable cures to patients. It is also being used to simulate patient data for simulated clinical trials and to test the accuracy of AI-based diagnostic tools. In retrieval-augmented generation, it is being used to produce better answers for staff in Q&A applications.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [22]:
# Your prompt here
prompt_2 = """
Whats the latest emerging AI startup innovations. Please give me the top 3, including company name, description, procut and sources.
Please display in a neatly formated markdown table.
"""


# Run it
response_2 = agent_executor.invoke({"input": prompt_2})





> Entering new AgentExecutor chain...
I need to search for the latest emerging AI startup innovations. I will use TavilySafeSearch to find this information. I will then format the information into a markdown table. 

Action: TavilySafeSearch
Action Input: Latest emerging AI startup innovations{'title': 'Latest AI News and AI Breakthroughs that Matter Most: 2026 & 2025', 'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates', 'content': 'Date: October 7, 2025  \nSummary: Italian foodtech startup Tuidi closed a €3 million seed round, led by Vertis SGR and Azimut, to scale its AI store-management platform, Delphi, across grocery chains. Delphi acts as a “store controller,” offering daily recommendations on procurement, pricing, assortment, and staffing. Early adopters have seen sales gains up to +2% and order cost reductions near 10%. Tuidi aims to bring Italy’s grocery sector into a data-first future by automating operations that remain largely manual today.  \nSource: EU-St

In [23]:
display(Markdown(response_2["output"]))


| Company Name | Description | Product | Source |
| --- | --- | --- | --- |
| Tuidi | Italian foodtech startup that has developed an AI store-management platform, Delphi, to automate operations in grocery chains. | Delphi, an AI store-management platform offering daily recommendations on procurement, pricing, assortment, and staffing. | [EU-Startups](https://www.crescendo.ai/news/latest-ai-news-and-updates) |
| Groq | AI chip startup that has entered into a strategic partnership with Nvidia, which includes a non-exclusive licensing deal for Groq’s AI inference chip technology. | AI inference chip technology. | [Crescendo](https://www.crescendo.ai/news/latest-ai-news-and-updates) |
| Anthropic | One of the hottest AI startups thanks to its innovative Claude family of LLMs. | Claude family of LLMs. | [CRN](https://www.crn.com/news/ai/2026/the-10-ai-startup-companies-to-watch-in-2026) |

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [24]:
# Your prompt here
prompt_3 = """
Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
Please display in a neatly formated markdown table.
"""

# Run it
response_3 = agent_executor.invoke({"input": prompt_3})




> Entering new AgentExecutor chain...
I need to find the key features, pricing, and reviews of both OpenAI’s ChatGPT Team and Anthropic’s Claude Pro. I will use TavilySafeSearch to find this information.

Action: TavilySafeSearch
Action Input: OpenAI’s ChatGPT Team key features, pricing, and reviews
{'title': 'ChatGPT Plans: Comparing Free, Plus, Pro, Business & Enterprise', 'url': 'https://intuitionlabs.ai/articles/chatgpt-plans-comparison', 'content': 'In January 2024, OpenAI filled the “middle” market with ChatGPT Teams (later renamed Business), a self-serve plan for small and mid-size teams (( "Highlights: Product")) (( "Highlights: How does the pricing work,for ChatGPT Business")). Teams offered collaboration (shared workspace, custom GPTs, admin tools) along with advanced models (GPT-4 with 32K context, premium DALL·E-3 images, browsing, data analysis) (( "Highlights: ChatGPT Team includes:")) (( "Highlights: ,to new features and improvements")). Business was priced per user ($

In [25]:
display(Markdown(response_3["output"]))


| | OpenAI’s ChatGPT Team | Anthropic’s Claude Pro |
|---|---|---|
| Key Features | - Access to advanced models like GPT-4 and DALL·E 3<br>- Collaborative workspace and admin tools<br>- Data privacy and control<br>- Customizable GPTs for specific needs | - 5x more usage than the free tier<br>- Priority access to Claude.ai during high-traffic periods<br>- Early access to new features |
| Pricing | - $25 per user per month when billed annually<br>- $30 per user per month when billed monthly | - $17 per month when billed annually<br>- $20 per month when billed monthly |
| Reviews | - Enhanced collaboration tools that make teamwork smoother<br>- Strong security features that keep your data safe<br>- Potential drawbacks include minimum two-seat requirement, restricted to OpenAI models | - Some users complain about the usage limits and customer service<br>- Others find the tool useful for productivity tasks and appreciate the priority access during peak times |

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


In [26]:
# Your prompt here
own_solution = """
What are the latest updates from Davos at the WEF in 2026 in terms of geopolitics?
Please display in a neatly formated markdown table.
"""

# Run it
response_4 = agent_executor.invoke({"input": own_solution})



> Entering new AgentExecutor chain...
I need to search for the latest updates from Davos in terms of geopolitics. I will use the TavilySafeSearch tool to find this information. 

Action: TavilySafeSearch
Action Input: Latest geopolitical updates from Davos
{'title': 'Dispatch From Davos: The Geopolitics of the New Realism', 'url': 'https://www.cfr.org/articles/dispatch-from-davos-the-geopolitics-of-the-new-realism', 'content': 'Every year, commentators try to assess the zeitgeist of Davos. My take is that people were on edge most of the week. The anxiety leading up to Trump’s speech on Wednesday was palpable, though his remarks produced a collective, if cautious, sigh of relief, particularly among the Europeans. And there were glimmers of optimism, particularly around artificial intelligence (AI). Witness the humanoid and canine-like robots walking along the Promenade. But unlike prior years when the focus was very much on the opportunity of technology, there was at least as much att

In [27]:
display(Markdown(response_4["output"]))

| Source | Key Points |
| --- | --- |
| [Dispatch From Davos: The Geopolitics of the New Realism](https://www.cfr.org/articles/dispatch-from-davos-the-geopolitics-of-the-new-realism) | Anxiety leading up to Trump’s speech, focus on geopolitical risk, scant attention to current geopolitics like Ukraine, Gaza, Iran, Venezuela, and Sudan, U.S.-China relationship largely absent from the agenda, climate change not front and center, European powers seeking to develop coalitions of the willing to forge a new geopolitical equilibrium. |
| [4 takeaways from Davos 2026: New deals, a reckoning, dialogue ...](https://www.weforum.org/stories/2026/01/4-takeaways-from-davos-2026/) | Tension between systemic fault lines and geopolitical noise. |
| [Five takeaways from Davos 2026 - Reuters](https://www.reuters.com/business/davos/five-takeaways-davos-2026-2026-01-23/) | Europe learned the value of standing up to Trump, Europe's trust in the transatlantic relationship with Washington has been badly rattled, Ukraine was overshadowed during Davos, peace deal appears elusive, Trump's unpredictability a defining feature of the event. |
| [At Davos, U.S. allies question a fraying world order - NPR](https://www.npr.org/2026/01/23/nx-s1-5686526/trump-davos-world-order) | Rules-based international order is fading, countries can no longer assume the United States would reliably act as the system's stabilizing force, growing uncertainty about America's role as a global leader, disruptive statements from President Trump unsettled global markets and strained relations with allies. |
| [Investors went to Davos for AI. They left talking about Greenland](https://www.cnbc.com/2026/01/24/davos-ai-greenland-trump-musk-energy-geopolitics.html) | Conversations ended up back at tariffs, Greenland, geopolitical tensions, global rules investors have relied on for decades are shifting, tension between swift innovation and political uncertainty. |